In [1]:
import duckdb
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
from folium.plugins import HeatMapWithTime
import seaborn as sns
import pandas as pd

In [ ]:
df = pd.read_parquet("pronostico_temperatura.parquet")

## Análisis de estaciones

En esta sección, visualizaremos en un principio todas las estaciones presentes de ecobicis

In [3]:
with duckdb.connect("modelo_estrella.duckdb", read_only=True) as con:
    dataframe_general = con.sql(
        """
        SELECT
	fr.fecha_origen_recorrido,
	fr.fecha_destino_recorrido,
	fr.duracion_estim_seg,
	fr.modelo_bicicleta,
	deo.nombre_estacion as estacion_origen,
	ded.nombre_estacion as estacion_destino,
	du.genero_usuario,
	du.edad_usuario,
	du.grupo_edad::text as grupo_edad,
	dc.date_key,
	dc.day_of_year,
	dc.week_key,
	dc.week_of_year,
	dc.day_of_week,
	dc.iso_day_of_week,
	dc.day_name,
	dc.first_day_of_week,
	dc.last_day_of_week,
	dc.month_key,
	dc.month_of_year,
	dc.day_of_month,
	dc.month_name_short,
	dc.month_name,
	dc.first_day_of_month,
	dc.last_day_of_month,
	dc.quarter_key,
	dc.quarter_of_year,
	dc.day_of_quarter,
	dc.quarter_desc_short,
	dc.quarter_desc,
	dc.first_day_of_quarter,
	dc.last_day_of_quarter,
	dc.year_key,
	dc.first_day_of_year,
	dc.last_day_of_year,
	dc.ordinal_weekday_of_month
FROM
	fact_recorridos fr
LEFT JOIN dim_estaciones deo ON
	fr.id_estacion_origen = deo.id_estacion
LEFT JOIN dim_estaciones ded ON
	fr.id_estacion_destino = ded.id_estacion
LEFT JOIN dim_usuario du ON
	fr.id_usuario = du.id_usuario
LEFT JOIN dim_calendar dc ON
	fr.fecha_origen_recorrido::date = dc.date_key
            """
    ).df() # Lo utilizamos para centrar el mapa

In [6]:
dataframe_general.to_parquet("dataset_entrenamiento.parquet")